<a href="https://colab.research.google.com/github/Yubraj45/telco-churn-pipeline/blob/main/Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [174]:
import pandas as pd

In [175]:
df= pd.read_csv('Churn.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [176]:
df.dtypes

,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


In [177]:
print(df['TotalCharges'].dtype)

object


In [178]:
print(df['Churn'].value_counts(normalize=True))
print(df.isnull().sum().sort_values(ascending=False).head(3))

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64
customerID       0
gender           0
SeniorCitizen    0
dtype: int64


In [179]:
problem_rows =  df[pd.to_numeric(df['TotalCharges'], errors='coerce').isna()]
print(f"Rows with non-numeric TotalCharges: {len(problem_rows)}")
print(problem_rows[['TotalCharges', 'tenure', 'MonthlyCharges']].head())

Rows with non-numeric TotalCharges: 11
     TotalCharges  tenure  MonthlyCharges
488                     0           52.55
753                     0           20.25
936                     0           80.85
1082                    0           25.75
1340                    0           56.05


In [180]:
import numpy as np

mask = pd.to_numeric(df['TotalCharges'], errors='coerce').isna()
print(df[mask][['TotalCharges', 'tenure', 'MonthlyCharges']])

     TotalCharges  tenure  MonthlyCharges
488                     0           52.55
753                     0           20.25
936                     0           80.85
1082                    0           25.75
1340                    0           56.05
3331                    0           19.85
3826                    0           25.35
4380                    0           20.00
5218                    0           19.70
6670                    0           73.35
6754                    0           61.90


In [181]:
df['TotalCharges']= df['TotalCharges'].fillna(0)

In [182]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
mask_zero_tenure = df['tenure']==0

#this fills the rows only where the tenure is ZERO
df.loc[mask_zero_tenure, 'TotalCharges'] = df.loc[mask_zero_tenure, 'TotalCharges'].fillna(0)

#for remaining Nan values
median_charges = df['TotalCharges'].median()
df['TotalCharges']= df['TotalCharges'].fillna(median_charges)

print('Remaining nan values :', df['TotalCharges'].isnull().sum())
print('The datatype of TotalCharges:', df['TotalCharges'].dtype)

Remaining nan values : 0
The datatype of TotalCharges: float64


In [183]:
df['Calculated_total']= df['tenure'] * df['MonthlyCharges']
df['Difference']= abs(df['TotalCharges'] - df['Calculated_total'])

print(f'Max difference is {df['Difference'].max()}')
print(f'Rows with difference >0.01 is {(df['Difference']>0.01).sum()}')
print(df[df['Difference'] > 0.01][['tenure', 'MonthlyCharges', 'TotalCharges', 'Calculated_total']].head())

Max difference is 373.2500000000009
Rows with difference >0.01 is 6418
   tenure  MonthlyCharges  TotalCharges  Calculated_total
1      34           56.95       1889.50            1936.3
2       2           53.85        108.15             107.7
3      45           42.30       1840.75            1903.5
4       2           70.70        151.65             141.4
5       8           99.65        820.50             797.2


In [184]:
print("Unique values in gender:", df['gender'].unique())
print("Unique values in PaymentMethod:", df['PaymentMethod'].unique())

Unique values in gender: ['Female' 'Male']
Unique values in PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']


In [185]:
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"Any nulls: {df['TotalCharges'].isnull().sum()}")
print(f"Any non-numeric: {pd.to_numeric(df['TotalCharges'], errors='coerce').isna().sum()}")

TotalCharges dtype: float64
Any nulls: 0
Any non-numeric: 0


In [186]:
df = df.drop('customerID', axis=1)
print(f"Final shape: {df.shape}")

Final shape: (7043, 22)


In [187]:
df.to_csv('telco_clean_final.csv', index=False)
print("Ready for production pipeline!")

Ready for production pipeline!


In [188]:
!mkdir -p telco_churn_pipeline
!ls

Churn.csv  sample_data	telco_churn_pipeline  telco_clean_final.csv


In [189]:
!mkdir -p telco_churn_pipeline/data
!mkdir -p telco_churn_pipeline/src
!mkdir -p telco_churn_pipeline/tests
!ls -la telco_churn_pipeline/

total 32
drwxr-xr-x 6 root root 4096 Jun  2 10:14 .
drwxr-xr-x 1 root root 4096 Jun  2 07:38 ..
drwxr-xr-x 5 root root 4096 Jun  2 08:11 data
-rw-r--r-- 1 root root  485 Jun  2 10:14 .gitignore
drwxr-xr-x 2 root root 4096 Jun  2 09:28 models
-rw-r--r-- 1 root root  818 Jun  2 10:14 README.md
drwxr-xr-x 2 root root 4096 Jun  2 10:13 src
drwxr-xr-x 2 root root 4096 Jun  2 10:13 tests


In [190]:
%%writefile telco_churn_pipeline/src/clean_data.py
def remove_customer_id(df):
  #removing the cuostmer id if there is one
  if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)
    print("removed customerID column")
  return df

Overwriting telco_churn_pipeline/src/clean_data.py


In [191]:
import sys
sys.path.append('/content/telco_churn_pipeline')

import pandas as pd
from src.clean_data import remove_customer_id

#createing test data
test_df= pd.DataFrame({
    'customerID':['A123', 'B456'],
    'Churn':['Yes', 'No']
})

print("Before:", test_df.columns.tolist())

cleaned_df= remove_customer_id(test_df)

print("after:", cleaned_df.columns.tolist())

Before: ['customerID', 'Churn']
after: ['Churn']


In [192]:
%%writefile telco_churn_pipeline/src/clean_data.py
def remove_customer_id(df):
  #removing the cuostmer id if there is one
  if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)
    print("removed customerID column")
  return df

def fix_total_charges(df):
  #convert totalcharges into numbers , and bad values becomes 0
  df= df.copy()
  df['TotalCharges']= pd.to_numeric(df['TotalCharges'], errors='coerce')
  df['TotalCharges']= df['TotalCharges'].fillna(0)
  print(f'Fixed TotalCharges dtype: {df["TotalCharges"].dtype}')
  return df


Overwriting telco_churn_pipeline/src/clean_data.py


In [193]:
import sys
sys.path.append('/content/telco_churn_pipeline')

import pandas as pd
from src.clean_data import remove_customer_id , fix_total_charges

# Create test data with messy TotalCharges
test_df = pd.DataFrame({
    'customerID': ['A123', 'B456', 'C789'],
    'tenure': [12, 0, 24],
    'TotalCharges': ['123.45', 'bad_data', '678.90'],
    'Churn': ['No', 'Yes', 'No']
})

print("Before fix:")
print(test_df[['TotalCharges', 'tenure']])
print(f"data type : {test_df['TotalCharges'].dtype}")

#APPLIYING FIXES
fixed_df= fix_total_charges(test_df)

print("\nAFTER fix:")
print(fixed_df[['TotalCharges', 'tenure']])
print(f"Data type: {fixed_df['TotalCharges'].dtype}")

Before fix:
  TotalCharges  tenure
0       123.45      12
1     bad_data       0
2       678.90      24
data type : object

AFTER fix:
   TotalCharges  tenure
0        123.45      12
1          0.00       0
2        678.90      24
Data type: float64


In [194]:
!cat telco_churn_pipeline/src/clean_data.py

def remove_customer_id(df):
  #removing the cuostmer id if there is one
  if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)
    print("removed customerID column")
  return df

def fix_total_charges(df):
  #convert totalcharges into numbers , and bad values becomes 0
  df= df.copy()
  df['TotalCharges']= pd.to_numeric(df['TotalCharges'], errors='coerce')
  df['TotalCharges']= df['TotalCharges'].fillna(0)
  print(f'Fixed TotalCharges dtype: {df["TotalCharges"].dtype}')
  return df


In [195]:
%%writefile telco_churn_pipeline/src/clean_data.py
import pandas as pd

def remove_customer_id(df):
    """Remove customerID column if it exists"""
    if 'customerID' in df.columns:
        df = df.drop('customerID', axis=1)
        print("Removed customerID column")
    return df

def fix_total_charges(df):
    """Convert TotalCharges to numbers, bad values become 0"""
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0)
    print(f"Fixed TotalCharges column. Data type: {df['TotalCharges'].dtype}")
    return df

Overwriting telco_churn_pipeline/src/clean_data.py


In [196]:
!cat telco_churn_pipeline/src/clean_data.py

import pandas as pd

def remove_customer_id(df):
    """Remove customerID column if it exists"""
    if 'customerID' in df.columns:
        df = df.drop('customerID', axis=1)
        print("Removed customerID column")
    return df

def fix_total_charges(df):
    """Convert TotalCharges to numbers, bad values become 0"""
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0)
    print(f"Fixed TotalCharges column. Data type: {df['TotalCharges'].dtype}")
    return df


In [197]:
!cat /content/telco_churn_pipeline/src/clean_data.py

import pandas as pd

def remove_customer_id(df):
    """Remove customerID column if it exists"""
    if 'customerID' in df.columns:
        df = df.drop('customerID', axis=1)
        print("Removed customerID column")
    return df

def fix_total_charges(df):
    """Convert TotalCharges to numbers, bad values become 0"""
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0)
    print(f"Fixed TotalCharges column. Data type: {df['TotalCharges'].dtype}")
    return df


In [198]:
import sys
import importlib
import pandas as pd

# Add path if not already there
if '/content/telco_churn_pipeline' not in sys.path:
    sys.path.append('/content/telco_churn_pipeline')

# Clear any cached version of the module
if 'src.clean_data' in sys.modules:
    del sys.modules['src.clean_data']
    print("Cleared old cached module")

# Import fresh
from src.clean_data import remove_customer_id, fix_total_charges

print("✅ Both functions imported successfully!")

# Test them
test_df = pd.DataFrame({
    'customerID': ['A123', 'B456'],
    'TotalCharges': ['123.45', 'bad_data'],
    'Churn': ['No', 'Yes']
})

print("\nTesting remove_customer_id:")
result1 = remove_customer_id(test_df)
print(f"Columns after remove: {result1.columns.tolist()}")

print("\nTesting fix_total_charges:")
result2 = fix_total_charges(test_df)
print(f"TotalCharges after fix: {result2['TotalCharges'].tolist()}")

Cleared old cached module
✅ Both functions imported successfully!

Testing remove_customer_id:
Removed customerID column
Columns after remove: ['TotalCharges', 'Churn']

Testing fix_total_charges:
Fixed TotalCharges column. Data type: float64
TotalCharges after fix: [123.45, 0.0]


In [199]:
import os

# Check if file exists
file_path = '/content/telco_churn_pipeline/src/clean_data.py'
print(f"File exists: {os.path.exists(file_path)}")

# List all files in the src folder
print("\nFiles in src folder:")
!ls -la /content/telco_churn_pipeline/src/

File exists: True

Files in src folder:
total 24
drwxr-xr-x 3 root root 4096 Jun  2 10:16 .
drwxr-xr-x 6 root root 4096 Jun  2 10:14 ..
-rw-r--r-- 1 root root  565 Jun  2 10:16 clean_data.py
-rw-r--r-- 1 root root  834 Jun  2 10:07 feature_engineering.py
-rw-r--r-- 1 root root 1020 Jun  2 10:07 prediction_pipeline.py
drwxr-xr-x 2 root root 4096 Jun  2 10:16 __pycache__


In [200]:
import sys
sys.path.append('/content/telco_churn_pipeline')

import pandas as pd
from src.clean_data import remove_customer_id, fix_total_charges

# Create test data with messy TotalCharges
test_df = pd.DataFrame({
    'customerID': ['A123', 'B456', 'C789'],
    'tenure': [12, 0, 24],
    'TotalCharges': ['123.45', 'bad_data', '678.90'],
    'Churn': ['No', 'Yes', 'No']
})

print("BEFORE fix:")
print(test_df[['TotalCharges', 'tenure']])
print(f"Data type: {test_df['TotalCharges'].dtype}")

# Apply our fix
fixed_df = fix_total_charges(test_df)

print("\nAFTER fix:")
print(fixed_df[['TotalCharges', 'tenure']])
print(f"Data type: {fixed_df['TotalCharges'].dtype}")

BEFORE fix:
  TotalCharges  tenure
0       123.45      12
1     bad_data       0
2       678.90      24
Data type: object
Fixed TotalCharges column. Data type: float64

AFTER fix:
   TotalCharges  tenure
0        123.45      12
1          0.00       0
2        678.90      24
Data type: float64


In [201]:
!cat telco_churn_pipeline/src/clean_data.py

import pandas as pd

def remove_customer_id(df):
    """Remove customerID column if it exists"""
    if 'customerID' in df.columns:
        df = df.drop('customerID', axis=1)
        print("Removed customerID column")
    return df

def fix_total_charges(df):
    """Convert TotalCharges to numbers, bad values become 0"""
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0)
    print(f"Fixed TotalCharges column. Data type: {df['TotalCharges'].dtype}")
    return df


In [202]:
# Write the complete, production-ready function
file_content = '''import pandas as pd
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def remove_customer_id(df):
    """Remove customerID column if it exists"""
    if 'customerID' in df.columns:
        df = df.drop('customerID', axis=1)
        logger.info("Removed customerID column")
    return df

def fix_total_charges(df):
    """Convert TotalCharges to numbers and apply business logic:
    - If tenure == 0, TotalCharges MUST be 0 (even if data says otherwise)
    - Otherwise, fill missing values with median
    """
    df = df.copy()

    # Convert to numeric (bad values become NaN)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

    # Count invalid values
    invalid_count = df['TotalCharges'].isna().sum()
    if invalid_count > 0:
        logger.warning(f"Found {invalid_count} invalid TotalCharges values")

    # BUSINESS RULE: tenure == 0 MUST have TotalCharges == 0
    mask_zero_tenure = df['tenure'] == 0
    zero_tenure_with_charges = mask_zero_tenure & (df['TotalCharges'] != 0)
    if zero_tenure_with_charges.any():
        logger.warning(f"Found {zero_tenure_with_charges.sum()} tenure=0 customers with non-zero charges. Correcting to 0.")

    df.loc[mask_zero_tenure, 'TotalCharges'] = 0

    # Fill remaining NaN values (tenure > 0) with median
    remaining_nulls = df['TotalCharges'].isna().sum()
    if remaining_nulls > 0:
        median_val = df['TotalCharges'].median()
        logger.info(f"Filling {remaining_nulls} missing values with median: {median_val}")
        df['TotalCharges'] = df['TotalCharges'].fillna(median_val)

    logger.info(f"TotalCharges fix complete. Data type: {df['TotalCharges'].dtype}")
    return df
'''

# Write the file
with open('/content/telco_churn_pipeline/src/clean_data.py', 'w') as f:
    f.write(file_content)

print("File updated with business logic!")

File updated with business logic!


In [203]:
import sys
sys.path.append('/content/telco_churn_pipeline')

if 'src.clean_data' in sys.modules:
  del sys.modules['src.clean_data']

from src.clean_data import fix_total_charges
import pandas as pd


test_df = pd.DataFrame({
    'tenure': [0,0,5,12,24],
    'TotalCharges': ['999.99', 'invalid', '50.0', None, '300.0']
})

print("Watch the LOGGING output below:")
result = fix_total_charges(test_df)

print("\n\nFinal result:")
print(result)


Watch the LOGGING output below:


Final result:
   tenure  TotalCharges
0       0           0.0
1       0           0.0
2       5          50.0
3      12          25.0
4      24         300.0


In [204]:
import logging

def demo_logging(log_level):
    """Show how different log levels filter messages"""
    # Clear existing handlers
    for handler in logging.root.handlers[:]:
        logging.root.removeiHandler(handler)

    # Set the log level
    logging.basicConfig(level=log_level, format='%(levelname)s: %(message)s')
    logger = logging.getLogger(__name__)

    print(f"\n--- Log level set to {log_level.__name__} ---")
    logger.debug("Debug message")
    logger.info("Info message")
    logger.warning("Warning message")
    logger.error("Error message")

In [205]:
import logging

def demonstrate_threshold(threshold_value, threshold_name):
    """Show how changing threshold filters messages"""

    # Clear existing handlers
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    # Set the threshold
    logging.basicConfig(level=threshold_value,
                        format='%(levelname)s: %(message)s')
    logger = logging.getLogger(__name__)

    print(f"\n{'='*50}")
    print(f"THRESHOLD SET TO: {threshold_name}")
    print(f"(Actual value: {threshold_value})")
    print(f"{'='*50}")

    # Send messages at all levels
    logger.debug("🔍 DEBUG: Detailed variable values")
    logger.info("ℹ️ INFO: Pipeline started normally")
    logger.warning("⚠️ WARNING: Found unexpected data")
    logger.error("❌ ERROR: Could not connect to database")
    logger.critical("💀 CRITICAL: System is crashing!")
    print(f"\n→ Only messages at {threshold_name} level and ABOVE are shown\n")

# Run the demo with correct parameters
demonstrate_threshold(logging.DEBUG, "DEBUG")
demonstrate_threshold(logging.WARNING, "WARNING")
demonstrate_threshold(logging.ERROR, "ERROR")

DEBUG: 🔍 DEBUG: Detailed variable values
INFO: ℹ️ INFO: Pipeline started normally
ERROR: ❌ ERROR: Could not connect to database
CRITICAL: 💀 CRITICAL: System is crashing!
ERROR: ❌ ERROR: Could not connect to database
CRITICAL: 💀 CRITICAL: System is crashing!
ERROR: ❌ ERROR: Could not connect to database
CRITICAL: 💀 CRITICAL: System is crashing!



THRESHOLD SET TO: DEBUG
(Actual value: 10)

→ Only messages at DEBUG level and ABOVE are shown


THRESHOLD SET TO: WARNING
(Actual value: 30)

→ Only messages at WARNING level and ABOVE are shown


THRESHOLD SET TO: ERROR
(Actual value: 40)

→ Only messages at ERROR level and ABOVE are shown



In [206]:
# Check what log levels we actually used
!grep "logger\." /content/telco_churn_pipeline/src/clean_data.py

        logger.info("Removed customerID column")
        logger.warning(f"Found {invalid_count} invalid TotalCharges values")
        logger.warning(f"Found {zero_tenure_with_charges.sum()} tenure=0 customers with non-zero charges. Correcting to 0.")
        logger.info(f"Filling {remaining_nulls} missing values with median: {median_val}")
    logger.info(f"TotalCharges fix complete. Data type: {df['TotalCharges'].dtype}")


In [207]:
%%writefile telco_churn_pipeline/tests/test_clean_data.py
"""
Unit tests for clean_data.py
Run with: pytest tests/test_clean_data.py -v
"""

import pytest
import pandas as pd
import sys
import os

# Add parent directory to path
sys.path.append('/content/telco_churn_pipeline')

from src.clean_data import remove_customer_id, fix_total_charges

def test_remove_customer_id():
    """Test that customerID column is removed"""
    df = pd.DataFrame({
        'customerID': ['A123', 'B456'],
        'Churn': ['Yes', 'No']
    })

    result = remove_customer_id(df)

    # Assertions - if these fail, test fails
    assert 'customerID' not in result.columns
    assert result.shape[1] == 1

def test_fix_total_charges_converts_to_numbers():
    """Test that string numbers become float"""
    df = pd.DataFrame({
        'tenure': [1, 2],
        'TotalCharges': ['123.45', '678.90']
    })

    result = fix_total_charges(df)

    assert result['TotalCharges'].dtype == 'float64'
    assert result['TotalCharges'].iloc[0] == 123.45

def test_fix_total_charges_handles_bad_data():
    """Test that invalid values become 0 for tenure=0 customers"""
    df = pd.DataFrame({
        'tenure': [0, 0],
        'TotalCharges': ['bad_data', 'also_invalid']
    })

    result = fix_total_charges(df)

    # tenure=0 customers should get 0
    assert result['TotalCharges'].iloc[0] == 0
    assert result['TotalCharges'].iloc[1] == 0

def test_fix_total_charges_tenure_zero_rule():
    """Test business rule: tenure=0 MUST have TotalCharges=0"""
    df = pd.DataFrame({
        'tenure': [0, 0, 1],
        'TotalCharges': ['999.99', 'invalid', '100.00']
    })

    result = fix_total_charges(df)

    # Both tenure=0 customers should be 0
    assert result['TotalCharges'].iloc[0] == 0
    assert result['TotalCharges'].iloc[1] == 0
    # tenure=1 customer should keep value
    assert result['TotalCharges'].iloc[2] == 100.00

print("Test file created successfully!")

Overwriting telco_churn_pipeline/tests/test_clean_data.py


In [208]:
!cd /content/telco_churn_pipeline && python -m pytest tests/test_clean_data.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/telco_churn_pipeline
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.5
collected 4 items                                                              

tests/test_clean_data.py::test_remove_customer_id PASSED                 [ 25%]
tests/test_clean_data.py::test_fix_total_charges_converts_to_numbers PASSED [ 50%]
tests/test_clean_data.py::test_fix_total_charges_handles_bad_data PASSED [ 75%]
tests/test_clean_data.py::test_fix_total_charges_tenure_zero_rule PASSED [100%]

============================== 4 passed in 0.62s ===============================


In [209]:
# Run this to celebrate your working tests!
!cd /content/telco_churn_pipeline && python -m pytest tests/test_clean_data.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/telco_churn_pipeline
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.5
collected 4 items                                                              

tests/test_clean_data.py::test_remove_customer_id PASSED                 [ 25%]
tests/test_clean_data.py::test_fix_total_charges_converts_to_numbers PASSED [ 50%]
tests/test_clean_data.py::test_fix_total_charges_handles_bad_data PASSED [ 75%]
tests/test_clean_data.py::test_fix_total_charges_tenure_zero_rule PASSED [100%]

============================== 4 passed in 0.57s ===============================


NOW WE WILL FINALLY START OUR PREDICTION PART , BUT FIRST WE WILL DO FEATURE ENGINEERING , PREPROCESSING AND ALL THE REQUIRED STEPS

In [210]:
import pandas as pd
import sys
sys.path.append('/content/telco_churn_pipeline')

from src.clean_data import remove_customer_id ,  fix_total_charges
raw_df = pd.read_csv('Churn.csv')
print(raw_df.shape)


(7043, 21)


In [211]:
cleaned_df= fix_total_charges(raw_df)
cleaned_df= remove_customer_id(cleaned_df)
cleaned_df.shape

(7043, 20)

In [212]:
os.makedirs('/content/telco_churn_pipeline/data/processed', exist_ok=True)

In [213]:
!ls -la /content/telco_churn_pipeline/data/


total 20
drwxr-xr-x 5 root root 4096 Jun  2 08:11  .
drwxr-xr-x 7 root root 4096 Jun  2 10:16  ..
drwxr-xr-x 2 root root 4096 Jun  2 08:11  processed
drwxr-xr-x 2 root root 4096 Jun  2 08:10 'Untitled Folder'
drwxr-xr-x 2 root root 4096 Jun  2 08:10 'Untitled Folder 1'


In [214]:
cleaned_df.to_csv('/content/telco_churn_pipeline/data/processed/telco_clean.csv', index=False)
print("Cleaned data saved")

Cleaned data saved


In [215]:
%%writefile /content/telco_churn_pipeline/src/feature_engineering.py
"""
Simple feature engineering for Telco Churn
"""

import pandas as pd
from sklearn.preprocessing import LabelEncoder

def encoded_categorical(df, columns_to_encode):
    """Convert categorical columns to numbers"""
    df = df.copy()
    encoders = {}

    for col in columns_to_encode:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
        print(f"  Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

    return df, encoders

def prepare_features(df, target_column='Churn'):
    """Split features and target"""
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    # Convert target to binary (Yes=1, No=0)
    y = (y == 'Yes').astype(int)

    return X, y

print("Feature engineering module created successfully!")

Overwriting /content/telco_churn_pipeline/src/feature_engineering.py


In [216]:
def prepare_features(df, target_column='Churn'):
  #splitting target and features
  X= df.drop(target_column, axis=1)
  y= df[target_column]

  y= (y=='Yes').astype(int)

  return X , y
print('Feature engineering modle created')

Feature engineering modle created


In [217]:
# Let's see the entire content of your feature_engineering.py
!cat /content/telco_churn_pipeline/src/feature_engineering.py

"""
Simple feature engineering for Telco Churn
"""

import pandas as pd
from sklearn.preprocessing import LabelEncoder

def encoded_categorical(df, columns_to_encode):
    """Convert categorical columns to numbers"""
    df = df.copy()
    encoders = {}
    
    for col in columns_to_encode:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
        print(f"  Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")
    
    return df, encoders

def prepare_features(df, target_column='Churn'):
    """Split features and target"""
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    
    # Convert target to binary (Yes=1, No=0)
    y = (y == 'Yes').astype(int)
    
    return X, y

print("Feature engineering module created successfully!")


In [218]:
import sys
sys.path.append('/content/telco_churn_pipeline')

# Clear cache
if 'src.feature_engineering' in sys.modules:
    del sys.modules['src.feature_engineering']

# Import with the CORRECT function name (encoded_categorical, not encode_categorical)
from src.feature_engineering import encoded_categorical, prepare_features

print("✅ Both functions imported successfully!")

Feature engineering module created successfully!
✅ Both functions imported successfully!


In [219]:
# Re-run feature engineering and training with proper encoder saving
import pandas as pd
import sys
sys.path.append('/content/telco_churn_pipeline')

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import joblib
import os

# Load cleaned data
df = pd.read_csv('/content/telco_churn_pipeline/data/processed/telco_clean.csv')
print(f"✅ Loaded {df.shape[0]} rows")

# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'Churn' in categorical_cols:
    categorical_cols.remove('Churn')
print(f"📊 Categorical columns: {categorical_cols}")

# Encode properly - fit on original data, save BEFORE transform
encoders = {}
df_encoded = df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    # Fit on original categories
    le.fit(df[col].astype(str))
    # Save the encoder
    encoders[col] = le
    # Transform the data
    df_encoded[col] = le.transform(df[col].astype(str))
    print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Prepare target
y = (df_encoded['Churn'] == 'Yes').astype(int)
X = df_encoded.drop('Churn', axis=1)

print(f"\n🔢 Features: {X.shape[1]}, Target: {y.value_counts().to_dict()}")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("🤖 Model trained!")

# Evaluate
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(f"\n📈 ROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.3f}")

# Save model and encoders
os.makedirs('/content/telco_churn_pipeline/models', exist_ok=True)
joblib.dump(model, '/content/telco_churn_pipeline/models/churn_model.pkl')
joblib.dump(encoders, '/content/telco_churn_pipeline/models/encoders.pkl')
print("\n💾 Model and encoders saved successfully!")

# Verify encoders now have proper categories
print("\n✅ Encoders now have correct categories:")
for col, encoder in encoders.items():
    print(f"  {col}: {encoder.classes_.tolist()[:3]}...")

✅ Loaded 7043 rows
📊 Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
  gender: {'Female': np.int64(0), 'Male': np.int64(1)}
  Partner: {'No': np.int64(0), 'Yes': np.int64(1)}
  Dependents: {'No': np.int64(0), 'Yes': np.int64(1)}
  PhoneService: {'No': np.int64(0), 'Yes': np.int64(1)}
  MultipleLines: {'No': np.int64(0), 'No phone service': np.int64(1), 'Yes': np.int64(2)}
  InternetService: {'DSL': np.int64(0), 'Fiber optic': np.int64(1), 'No': np.int64(2)}
  OnlineSecurity: {'No': np.int64(0), 'No internet service': np.int64(1), 'Yes': np.int64(2)}
  OnlineBackup: {'No': np.int64(0), 'No internet service': np.int64(1), 'Yes': np.int64(2)}
  DeviceProtection: {'No': np.int64(0), 'No internet service': np.int64(1), 'Yes': np.int64(2)}
  TechSupport: {'No': np.int64(0), 'No inte

In [220]:
X_train, X_test , y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("X_train shape:",X_train.shape[0])
print("X_test shape:",X_test.shape[0])

X_train shape: 5634
X_test shape: 1409


In [221]:
#TRAIN MODEL
model= RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model Trained")

Model Trained


In [222]:
#Evaluation of the model
from sklearn.metrics import roc_auc_score, classification_report
y_pred = model.predict(X_test)
Y_pred_prob = model.predict_proba(X_test)[:,1]

print("\n" + "="*50)
print("Model Performance:")
print("="*50)
print(f"ROC AUC Score: {roc_auc_score(y_test, Y_pred_prob):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Stays', 'Churns']))



Model Performance:
ROC AUC Score: 0.825

Classification Report:
              precision    recall  f1-score   support

       Stays       0.84      0.90      0.87      1035
      Churns       0.64      0.51      0.57       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.78      0.79      0.79      1409



In [223]:
import os

# Create the models folder
os.makedirs('/content/telco_churn_pipeline/models', exist_ok=True)
print("✅ Created models folder")

# Verify it exists
!ls -la /content/telco_churn_pipeline/

✅ Created models folder
total 36
drwxr-xr-x 7 root root 4096 Jun  2 10:16 .
drwxr-xr-x 1 root root 4096 Jun  2 07:38 ..
drwxr-xr-x 5 root root 4096 Jun  2 08:11 data
-rw-r--r-- 1 root root  485 Jun  2 10:14 .gitignore
drwxr-xr-x 2 root root 4096 Jun  2 09:28 models
drwxr-xr-x 3 root root 4096 Jun  2 10:16 .pytest_cache
-rw-r--r-- 1 root root  818 Jun  2 10:14 README.md
drwxr-xr-x 3 root root 4096 Jun  2 10:16 src
drwxr-xr-x 3 root root 4096 Jun  2 10:16 tests


In [224]:
#Saving
joblib.dump(model, '/content/telco_churn_pipeline/models/churn_model.pkl')
joblib.dump(encoders, '/content/telco_churn_pipeline/models/encoders.pkl')
print("\n Model and encoders saved successfully!")


 Model and encoders saved successfully!


In [225]:
#Prediction Pipleline

%%writefile /content/telco_churn_pipeline/src/prediction_pipeline.py

# production prediction module

import pandas as pd
import sys
import joblib

class ChurnPredictor:
  def __init__(self, model_path='/content/telco_churn_pipeline/models/churn_model.pkl',
                 encoder_path='/content/telco_churn_pipeline/models/encoders.pkl'):
    self.model= joblib.load(model_path)
    self.encoders= joblib.load(encoder_path)
    print("Predictor loaded and ready")

  def predict(self, customer_data):
    #predict Churn for simple customer
    df= pd.DataFrame([customer_data])

    #Encode categorical columns
    for col, encoder in self.encoders.items():
      if col in df.columns:
        df[col] = encoder.transform(df[col].astype(str))

    #make prediction
    proba = self.model.predict_proba(df)[0,1]
    pred= self.model.predict(df)[0]

    return{
        'churn_probability': round(proba * 100, 2),
            'will_churn': bool(pred),
            'risk_level': 'HIGH' if proba > 0.7 else 'MEDIUM' if proba > 0.3 else 'LOW'
    }

print("Prediction module created")

Overwriting /content/telco_churn_pipeline/src/prediction_pipeline.py


In [226]:
from src.prediction_pipeline import ChurnPredictor

#INITIALIZING PREDICTOR
predictor= ChurnPredictor()

#TESTING PREDICTION
high_risk_customer = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 2,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',  # HIGH RISK!
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 85.5,
    'TotalCharges': 171.0
}


#GET PREDICTION
result = predictor.predict(high_risk_customer)

print("\n" + "="*50)
print("🔮 CHURN PREDICTION RESULT")
print("="*50)
print(f"📊 Churn Probability: {result['churn_probability']}%")
print(f"⚠️ Will Churn: {result['will_churn']}")
print(f"🎯 Risk Level: {result['risk_level']}")
print("="*50)

if result['will_churn']:
  print("\n RECOMMENDATION: Offer discount or upgrade to long_term contract")
else:
  print("\n RECOMMENDATION: Customer likely to stay _ low priority")

Predictor loaded and ready

🔮 CHURN PREDICTION RESULT
📊 Churn Probability: 79.0%
⚠️ Will Churn: True
🎯 Risk Level: HIGH

 RECOMMENDATION: Offer discount or upgrade to long_term contract


In [227]:
!tree /content/telco_churn_pipeline -L 2

/bin/bash: line 1: tree: command not found


In [228]:
!ls -la /content/telco_churn_pipeline/
!ls -la /content/telco_churn_pipeline/src/
!ls -la /content/telco_churn_pipeline/models/
!ls -la /content/telco_churn_pipeline/data/processed/

total 36
drwxr-xr-x 7 root root 4096 Jun  2 10:16 .
drwxr-xr-x 1 root root 4096 Jun  2 07:38 ..
drwxr-xr-x 5 root root 4096 Jun  2 08:11 data
-rw-r--r-- 1 root root  485 Jun  2 10:14 .gitignore
drwxr-xr-x 2 root root 4096 Jun  2 09:28 models
drwxr-xr-x 3 root root 4096 Jun  2 10:16 .pytest_cache
-rw-r--r-- 1 root root  818 Jun  2 10:14 README.md
drwxr-xr-x 3 root root 4096 Jun  2 10:16 src
drwxr-xr-x 3 root root 4096 Jun  2 10:16 tests
total 24
drwxr-xr-x 3 root root 4096 Jun  2 10:16 .
drwxr-xr-x 7 root root 4096 Jun  2 10:16 ..
-rw-r--r-- 1 root root 1765 Jun  2 10:16 clean_data.py
-rw-r--r-- 1 root root  834 Jun  2 10:16 feature_engineering.py
-rw-r--r-- 1 root root 1020 Jun  2 10:16 prediction_pipeline.py
drwxr-xr-x 2 root root 4096 Jun  2 10:16 __pycache__
total 17236
drwxr-xr-x 2 root root     4096 Jun  2 09:28 .
drwxr-xr-x 7 root root     4096 Jun  2 10:16 ..
-rw-r--r-- 1 root root 17636553 Jun  2 10:16 churn_model.pkl
-rw-r--r-- 1 root root     4067 Jun  2 10:16 encoders.pkl
to

In [229]:
# Count your lines of code
print("📊 Code Statistics:")
print("="*40)

# Count lines in each Python file
for file in ['clean_data.py', 'feature_engineering.py', 'prediction_pipeline.py']:
    try:
        with open(f'/content/telco_churn_pipeline/src/{file}', 'r') as f:
            lines = len(f.readlines())
            print(f"📝 {file}: {lines} lines")
    except:
        print(f"❌ {file}: not found")

print("="*40)
print("🎉 Total: Production-ready ML pipeline!")

📊 Code Statistics:
📝 clean_data.py: 46 lines
📝 feature_engineering.py: 31 lines
📝 prediction_pipeline.py: 34 lines
🎉 Total: Production-ready ML pipeline!


In [230]:
# Remove any temporary or cache files
!cd /content/telco_churn_pipeline && rm -rf __pycache__ .pytest_cache
!cd /content/telco_churn_pipeline/src && rm -rf __pycache__
!cd /content/telco_churn_pipeline/tests && rm -rf __pycache__

print("✅ Cleaned cache files")

✅ Cleaned cache files
